# 3. Hipnogramas — cohorte unificada

Dibuja un hipnograma por paciente a partir de **dataset/epocas_unificado.csv**: la curva de fases a lo largo de la noche, con la misma paleta y el mismo colorbar para las tres bases.

**Salida:** **imagenes/hipnogramas/hipnograma_<paciente>.png**.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path("..").resolve()
RUTA_CSV = RAIZ / "dataset/epocas_unificado.csv"
CARPETA_OUT = RAIZ / "imagenes/hipnogramas"
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

FASES = {0: "Vigilia", 1: "S1", 2: "S2", 3: "SWS", 4: "REM"}  # 5=Sin_clasificar oculto
SLP_RGB = [
    (230/255, 230/255, 250/255),  # Lavender    - Vigilia
    (173/255, 216/255, 230/255),  # Light-Blue  - S1
    (173/255, 255/255,  47/255),  # GreenYellow - S2
    (255/255, 127/255,  80/255),  # Coral       - SWS
    ( 65/255, 105/255, 225/255),  # RoyalBlue   - REM
]

df = pd.read_csv(RUTA_CSV)
print("Pacientes:", df['paciente'].nunique())

Pacientes: 127


In [2]:
def graficar_hipnograma(g, paciente, out_path):
    g = g[g['fase_num'].between(0, 4)].copy()  # solo las 5 fases fisiológicas
    if g.empty:
        return False
    g['horas'] = g['epoca'] / 120.0

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(g['horas'], g['fase_num'], color='k', linewidth=0.8)
    ax.scatter(g['horas'], g['fase_num'], s=10,
               color=[SLP_RGB[i] for i in g['fase_num']])
    ax.set_title(f"Hipnograma — {paciente}")
    ax.set_xlabel("Horas")
    ax.set_yticks(list(FASES.keys()))
    ax.set_yticklabels(list(FASES.values()))
    ax.set_ylim(-0.1, 4.1)
    ax.grid(color='gray', linestyle='--', linewidth=0.4, axis='y')

    cmap = plt.cm.colors.ListedColormap(SLP_RGB)
    bounds = np.linspace(-0.5, 4.5, 6)
    norm = plt.cm.colors.BoundaryNorm(bounds, cmap.N)
    cb = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm),
                       ax=ax, orientation='horizontal', pad=0.18, shrink=0.6)
    cb.set_ticks(np.arange(0, 5))
    cb.set_ticklabels(list(FASES.values()))

    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.close(fig)
    return True

n = 0
for paciente, g in df.groupby('paciente'):
    out = CARPETA_OUT / f"hipnograma_{paciente}.png"
    if graficar_hipnograma(g, paciente, out):
        n += 1
print(f"Generados {n} hipnogramas en {CARPETA_OUT}")

Generados 127 hipnogramas en imagenes/hipnogramas


## Hipnograma modelo para el marco teórico

Versión depurada de un único participante sano representativo (**CAP_S12_N1**), usada como figura ilustrativa de la arquitectura cíclica del sueño en el Capítulo 2.

In [ ]:
# === Hipnograma modelo (participante sano representativo) para el marco teórico ===
# Se selecciona CAP_S12_N1 por mostrar la arquitectura canónica: SWS concentrado en
# el primer tercio, episodios de REM que se alargan hacia la mañana y ~5 ciclos limpios.
PAC_MODELO = "CAP_S12_N1"
gm = df[df['paciente'] == PAC_MODELO].sort_values('epoca')
gm = gm[gm['fase_num'].between(0, 4)].copy()  # solo las 5 fases fisiológicas
gm['horas'] = gm['epoca'] / 120.0

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(gm['horas'], gm['fase_num'], color='k', linewidth=0.9)
ax.scatter(gm['horas'], gm['fase_num'], s=12, color=[SLP_RGB[i] for i in gm['fase_num']], zorder=3)
ax.set_title("Hipnograma de un participante sano representativo")
ax.set_xlabel("Tiempo (horas)")
ax.set_yticks(list(FASES.keys()))
ax.set_yticklabels(list(FASES.values()))
ax.set_ylim(-0.1, 4.1)
ax.grid(color='gray', linestyle='--', linewidth=0.4, axis='y')

cmap = plt.cm.colors.ListedColormap(SLP_RGB)
norm = plt.cm.colors.BoundaryNorm(np.linspace(-0.5, 4.5, 6), cmap.N)
cb = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
                  orientation='horizontal', pad=0.18, shrink=0.6)
cb.set_ticks(np.arange(0, 5)); cb.set_ticklabels(list(FASES.values()))
fig.tight_layout()
fig.savefig(CARPETA_OUT / "hipnograma_modelo.png", dpi=300)
plt.show()
print("Hipnograma modelo guardado:", CARPETA_OUT / "hipnograma_modelo.png")
